In [1]:
import tempfile
import os
from ansys.aedt.core import Desktop, generate_unique_name
import optuna
import utils
import logging
import sys
import numpy as np
import maker
from utils.path import create_path_matrix

c:\Users\luis.costa\.conda\envs\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
optuna.logging.get_logger("optuna").addHandler(logging.StreamHandler(sys.stdout))
study_name = "Optimization_filter" 
storage_name = "sqlite:///{}.db".format(study_name)

In [3]:
tmpfold = tempfile.gettempdir()

temp_folder = os.path.join(
    tmpfold,
    generate_unique_name("Combinator")
)

if not os.path.exists(temp_folder):
    os.mkdir(temp_folder)

non_graphical = os.getenv(
    "PYAEDT_NON_GRAPHICAL",
    "False"
).lower() in ("true", "1", "t")

d = Desktop(
    "2025.1",
    non_graphical=non_graphical,
    new_desktop=True
)


PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].


PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\LUIS~1.COS\AppData\Local\Temp\pyaedt_luis.costa_8a7b950f-da68-4011-ada4-4afc3b3785e3.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: New AEDT session is starting on gRPC port 62918.
PyAEDT INFO: Electronics Desktop started on gRPC port: 62918 after 27.478684663772583 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v251\AnsysEM
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.1 version started with process ID 11036.


In [4]:
matrix_size = 100

x_substrate = 202
y_substrate = 280

width_port = 5
height = 1.65

position_X = 101  
position_X2 = 0
position_Y = -140

initial_position1 = [0, (matrix_size - 1)]
final_position1 = [(matrix_size - 1), 0]

initial_position2 = [0,0]
final_position2 = [(matrix_size - 1), (matrix_size - 1)]

In [5]:
matrix1 = create_path_matrix(matrix_size, initial_position1, final_position1)
matrix2 = create_path_matrix(matrix_size, initial_position2, final_position2)   

In [6]:
def objective(trial):
    
    action = trial.suggest_int("action", 0, 9)

    #action = trial.suggest_int("action", 0, 9)
    env = maker.CombinerEnv(x_substrate, y_substrate, width_port, height, matrix_size, position_X, position_X2, position_Y, matrix1, matrix2)
    env.step(action)
    #observation = env.reset()

    return trial

study = optuna.create_study(
    directions=["minimize"],    
    #study_name=study_name, 
    #storage=storage_name, 
    #load_if_exists=True
)



[I 2025-11-18 11:31:06,531] A new study created in memory with name: no-name-693f1679-f906-4a3f-9ca8-ef4fb40a9d2d


A new study created in memory with name: no-name-693f1679-f906-4a3f-9ca8-ef4fb40a9d2d


In [ ]:
study.optimize(objective, n_trials=1)

PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 11036!


PyAEDT INFO: Project Project265 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_0OI' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 1sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec


In [ ]:
#d.release_desktop()